# 🥇 Classification de Footballeurs par CNN — Soulier d'Or
**ENSA de Fès — Module : DL & NLP — Année 2025/2026**

**Auteur :** Ayman Lebbar  Ismail mouass
**Encadrant :** Pr. Oussama EL GANNOUR

---
### Pipeline du projet
1. Installation des dépendances
2. Téléchargement du dataset (Kaggle)
3. Exploration et visualisation
4. Détection faciale (MTCNN) — création du dataset visages
5. Générateurs de données avec augmentation
6. Construction du modèle MobileNetV2 (Transfer Learning)
7. Entraînement Phase 1 (base gelée)
8. Entraînement Phase 2 (fine-tuning)
9. Sauvegarde sur Google Drive
10. Interface de démonstration (Gradio)

## Étape 1 — Installation des dépendances

In [ ]:
# Installation des dépendances
# IMPORTANT : après exécution, redémarrer le runtime (Runtime > Redémarrer la session)
!pip install lz4 mtcnn gradio -q
print('✅ Installation terminée — redémarre le runtime maintenant !')

## Étape 2 — Vérification GPU et imports

In [ ]:
# Vérification GPU et imports globaux
import os, json, zipfile, shutil, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
import tensorflow as tf
import cv2
from PIL import Image

warnings.filterwarnings('ignore')
tf.random.set_seed(42)

print('TensorFlow :', tf.__version__)
print('GPU actif  :', tf.config.list_physical_devices('GPU'))

# Paramètres globaux
IMG_SIZE    = (160, 160)
BATCH_SIZE  = 64
NUM_CLASSES = 22
SEED        = 42

## Étape 3 — Téléchargement du dataset Kaggle

In [ ]:
# Configuration Kaggle et téléchargement du dataset Golden Foot
KAGGLE_USERNAME = "VOTRE_USERNAME_KAGGLE"  # ← remplacer
KAGGLE_KEY      = "VOTRE_CLE_KAGGLE"    # ← remplacer

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

os.system('kaggle datasets download -d balabaskar/golden-foot-football-players-image-dataset -p /content/')

with zipfile.ZipFile('/content/golden-foot-football-players-image-dataset.zip', 'r') as z:
    z.extractall('/content/golden_foot_dataset')

DATASET_PATH = '/content/golden_foot_dataset/football_golden_foot/football_golden_foot'
print('✅ Dataset prêt !')

## Étape 4 — Exploration et visualisation du dataset

In [ ]:
# Exploration : distribution des classes
classes = sorted(os.listdir(DATASET_PATH))
counts  = {c: len(os.listdir(os.path.join(DATASET_PATH, c))) for c in classes}

print(f'Nombre de joueurs (classes) : {len(classes)}')
print(f'Nombre total d\'images      : {sum(counts.values())}')
print(f'Moyenne images/joueur       : {sum(counts.values())//len(classes)}')
print()
for c, n in counts.items():
    barre = '█' * int(n / 15)
    print(f'  {c:<30} {barre} {n}')

In [ ]:
# Visualisation — aperçu du dataset (une image par joueur)
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
fig.suptitle("Aperçu du dataset — Soulier d'Or (22 joueurs)", fontsize=14, fontweight='bold')

for ax, cls in zip(axes.flatten(), classes):
    img_file = random.choice(os.listdir(os.path.join(DATASET_PATH, cls)))
    img = mpimg.imread(os.path.join(DATASET_PATH, cls, img_file))
    ax.imshow(img)
    ax.set_title(cls.replace('_', ' ').title(), fontsize=7)
    ax.axis('off')

for ax in axes.flatten()[22:]:
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/apercu_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Graphique de distribution des classes
fig, ax = plt.subplots(figsize=(14, 6))
noms_court = [c.replace('_', ' ').title() for c in classes]
valeurs    = [counts[c] for c in classes]
colors     = plt.cm.Set3.colors[:len(classes)]

bars = ax.barh(noms_court, valeurs, color=colors, edgecolor='white', height=0.7)
ax.set_xlabel("Nombre d'images")
ax.set_title("Distribution des images par joueur", fontsize=13, fontweight='bold')
ax.axvline(x=sum(valeurs)/len(valeurs), color='red', linestyle='--',
           linewidth=1.5, label=f'Moyenne : {sum(valeurs)//len(valeurs)} img')
ax.legend()
for bar, val in zip(bars, valeurs):
    ax.text(bar.get_width()+2, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('/content/distribution_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## Étape 5 — Détection faciale avec MTCNN

In [ ]:
# Détection et recadrage des visages avec MTCNN
# Objectif : éliminer les artefacts (maillots, logos) pour focaliser sur le visage
import lz4
from mtcnn import MTCNN

detector      = MTCNN()
DATASET_FACES = '/content/golden_foot_faces'

def extraire_visage(img_path, target_size=(160, 160)):
    """Détecte et recadre le visage. Retourne l'image entière si aucun visage trouvé."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    resultats = detector.detect_faces(img_rgb)

    if resultats:
        best     = max(resultats, key=lambda x: x['confidence'])
        x, y, w, h = best['box']
        marge    = int(0.35 * max(w, h))
        x1 = max(0, x - marge)
        y1 = max(0, y - marge)
        x2 = min(img_rgb.shape[1], x + w + marge)
        y2 = min(img_rgb.shape[0], y + h + marge)
        region = img_rgb[y1:y2, x1:x2]
    else:
        region = img_rgb

    return Image.fromarray(region).resize(target_size)

# Test visuel sur Cristiano Ronaldo
test_dir = os.path.join(DATASET_PATH, 'cristiano_ronaldo')
test_img = os.path.join(test_dir, os.listdir(test_dir)[0])
visage   = extraire_visage(test_img)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(plt.imread(test_img))
axes[0].set_title('Image originale')
axes[0].axis('off')
axes[1].imshow(visage)
axes[1].set_title('Visage extrait (MTCNN)')
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/content/exemple_detection_faciale.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Détection faciale OK !')

In [ ]:
# Création du dataset de visages recadrés
if os.path.exists(DATASET_FACES):
    shutil.rmtree(DATASET_FACES)
os.makedirs(DATASET_FACES)

total_ok, total_skip = 0, 0

for cls in classes:
    src = os.path.join(DATASET_PATH, cls)
    dst = os.path.join(DATASET_FACES, cls)
    os.makedirs(dst, exist_ok=True)
    imgs = os.listdir(src)
    print(f'  {cls:<30} ({len(imgs)} imgs)...', end=' ')
    ok = 0
    for img_file in imgs:
        try:
            v = extraire_visage(os.path.join(src, img_file))
            if v:
                v.save(os.path.join(dst, img_file), 'JPEG', quality=90)
                ok += 1
        except:
            total_skip += 1
    total_ok += ok
    print(f'✓ {ok}')

print(f'\n✅ Dataset visages créé : {total_ok} images | Ignorées : {total_skip}')

## Étape 6 — Générateurs de données avec augmentation

In [ ]:
# Générateurs de données : augmentation sur train, normalisation seule sur val
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    # Augmentations géométriques
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.1,
    # Augmentations photométriques
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_FACES, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=SEED
)
val_gen = val_datagen.flow_from_directory(
    DATASET_FACES, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=SEED
)

# Sauvegarder le mapping classes ↔ index
class_names = {str(v): k for k, v in train_gen.class_indices.items()}
with open('/content/class_names.json', 'w') as f:
    json.dump(class_names, f)

print(f'Train : {train_gen.samples} images | Val : {val_gen.samples} images')
print(f'Classes : {train_gen.num_classes}')

## Étape 7 — Architecture du modèle (MobileNetV2 + Transfer Learning)

In [ ]:
# Construction du modèle : MobileNetV2 comme extracteur de features + tête personnalisée
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, regularizers
from tensorflow.keras.optimizers import Adam

# Base pré-entraînée sur ImageNet (gelée en Phase 1)
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(160, 160, 3))
base.trainable = False

# Tête de classification personnalisée
inputs  = tf.keras.Input(shape=(160, 160, 3))
x       = base(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(256, kernel_regularizer=regularizers.l2(0.005))(x)
x       = layers.BatchNormalization()(x)
x       = layers.Activation('relu')(x)
x       = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# Compilation avec Label Smoothing (évite la sur-confiance)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

total     = model.count_params()
trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'Paramètres total        : {total:,}')
print(f'Paramètres entraînables : {trainable:,}')
print(f'Paramètres gelés        : {total - trainable:,}')
model.summary()

## Étape 8 — Entraînement Phase 1 : base gelée (~5 min)

In [ ]:
# Callbacks : sauvegarde, arrêt anticipé, réduction du LR
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

os.makedirs('/content/checkpoints', exist_ok=True)

def get_callbacks(nom):
    return [
        ModelCheckpoint(
            f'/content/checkpoints/{nom}.keras',
            monitor='val_accuracy', save_best_only=True, verbose=1
        ),
        EarlyStopping(
            monitor='val_accuracy', patience=8,
            restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.3, patience=3,
            min_lr=1e-8, verbose=1
        )
    ]
print('✅ Callbacks prêts !')

In [ ]:
# Phase 1 : entraînement de la tête de classification (base MobileNetV2 gelée)
print('=' * 55)
print('  PHASE 1 — Base gelée (LR=1e-3) — ~5 min')
print('=' * 55)

h1 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=get_callbacks('phase1_best'),
    verbose=1
)

best_p1 = max(h1.history['val_accuracy'])
print(f'\n✅ Meilleure val_accuracy Phase 1 : {best_p1:.4f}')

## Étape 9 — Entraînement Phase 2 : fine-tuning (~10 min)

In [ ]:
# Phase 2 : fine-tuning des 54 dernières couches de MobileNetV2
print('=' * 55)
print('  PHASE 2 — Fine-tuning (LR=5e-5) — ~10 min')
print('=' * 55)

base.trainable = True
for layer in base.layers[:100]:  # Geler les 100 premières couches
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

trainable2 = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'Paramètres entraînables après déblocage : {trainable2:,}')

h2 = model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=get_callbacks('phase2_best'),
    verbose=1
)

best_p2 = max(h2.history['val_accuracy'])
print(f'\n✅ Meilleure val_accuracy Phase 2 : {best_p2:.4f}')
print(f'   Gain vs Phase 1               : +{best_p2 - best_p1:.4f}')

## Étape 10 — Courbes d'apprentissage et évaluation

In [ ]:
# Courbes d'apprentissage complètes (Phase 1 + Phase 2)
def merge(h1, h2):
    return {k: h1.history[k] + h2.history[k] for k in h1.history}

hall = merge(h1, h2)
sep  = len(h1.history['accuracy'])
ep   = range(1, len(hall['accuracy']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Courbes d'apprentissage — CNN Soulier d'Or", fontsize=14, fontweight='bold')

for ax, (tk, vk, titre) in zip(axes, [
    ('accuracy',  'val_accuracy',  'Accuracy'),
    ('loss',      'val_loss',      'Loss'),
    ('top3_acc',  'val_top3_acc',  'Top-3 Accuracy')
]):
    ax.plot(ep, hall[tk], label='Train', color='#2196F3', linewidth=2)
    ax.plot(ep, hall[vk], label='Val',   color='#FF9800', linewidth=2)
    ax.axvline(x=sep, color='red', linestyle='--', linewidth=1.5, label='Fine-tuning')
    ax.set_title(titre)
    ax.set_xlabel('Époque')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/courbes_apprentissage.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Résumé final :')
print(f'  val_accuracy   : {best_p2:.4f}')
print(f'  val_top3_acc   : {max(h2.history["val_top3_acc"]):.4f}')

In [ ]:
# Matrice de confusion
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = val_gen.classes
noms_classes = [k.replace('_', ' ').title() for k in sorted(train_gen.class_indices.keys())]

cm  = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(16, 14))
ConfusionMatrixDisplay(cm, display_labels=noms_classes).plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title("Matrice de confusion — CNN Soulier d'Or", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/matrice_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nRapport de classification :')
print(classification_report(y_true, y_pred, target_names=noms_classes, digits=3))

## Étape 11 — Sauvegarde sur Google Drive

In [ ]:
# Sauvegarde du modèle et des fichiers sur Google Drive
from google.colab import drive
drive.mount('/drive')

DRIVE_PATH = '/drive/MyDrive/soulier_dor_CNN'
os.makedirs(DRIVE_PATH, exist_ok=True)

fichiers = {
    '/content/checkpoints/phase2_best.keras'  : 'phase2_best.keras',
    '/content/class_names.json'                : 'class_names.json',
    '/content/courbes_apprentissage.png'        : 'courbes_apprentissage.png',
    '/content/matrice_confusion.png'            : 'matrice_confusion.png',
    '/content/apercu_dataset.png'               : 'apercu_dataset.png',
    '/content/distribution_classes.png'         : 'distribution_classes.png',
    '/content/exemple_detection_faciale.png'    : 'exemple_detection_faciale.png',
}

for src, dst in fichiers.items():
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_PATH}/{dst}')
        print(f'  ✓ {dst}')

print('\n✅ Tout sauvegardé sur Google Drive !')

## Étape 12 — Interface de démonstration Gradio

In [ ]:
# Interface Gradio avec détection faciale automatique + TTA + seuil de confiance
import gradio as gr
from tensorflow.keras.preprocessing.image import ImageDataGenerator

detector_g      = MTCNN()
SEUIL_CONFIANCE = 0.40
noms = [class_names[str(i)].replace('_', ' ').title() for i in range(len(class_names))]

def extraire_region(image_np):
    res = detector_g.detect_faces(image_np)
    if res:
        best       = max(res, key=lambda x: x['confidence'])
        x, y, w, h = best['box']
        marge      = int(0.35 * max(w, h))
        x1 = max(0, x - marge)
        y1 = max(0, y - marge)
        x2 = min(image_np.shape[1], x + w + marge)
        y2 = min(image_np.shape[0], y + h + marge)
        return image_np[y1:y2, x1:x2], True
    return image_np, False

def tta_predict(region_np, n=5):
    """Test Time Augmentation : moyenne de N prédictions légèrement augmentées."""
    aug = ImageDataGenerator(
        rotation_range=10, width_shift_range=0.05, height_shift_range=0.05,
        horizontal_flip=True, zoom_range=0.1, brightness_range=[0.9, 1.1],
        preprocessing_function=preprocess_input
    )
    img_arr = np.expand_dims(
        np.array(Image.fromarray(region_np).resize((160, 160)), dtype=np.float32), 0)
    preds = [model.predict(preprocess_input(img_arr.copy()), verbose=0)[0]]
    gen   = aug.flow(img_arr, batch_size=1)
    for _ in range(n - 1):
        preds.append(model.predict(next(gen), verbose=0)[0])
    return np.mean(preds, axis=0)

def predire(image):
    img_rgb        = np.array(Image.fromarray(image).convert('RGB'))
    region, trouve = extraire_region(img_rgb)
    note           = '✅ Visage détecté' if trouve else '⚠️ Image entière utilisée'

    pred = tta_predict(region, n=5)
    top5 = np.argsort(pred)[::-1][:5]
    conf = pred[top5[0]]

    if conf < SEUIL_CONFIANCE:
        texte = (f'{note}\n\n❓ Joueur non identifié avec certitude\n'
                 f'Confiance max : {conf*100:.1f}%\n\nCandidats :\n')
        for i, idx in enumerate(top5[:3]):
            texte += f'  {i+1}. {noms[idx]:<25} {pred[idx]*100:.1f}%\n'
    else:
        texte  = f'{note}\n\n🏆 {noms[top5[0]]}\n'
        texte += f'Confiance : {conf*100:.1f}%\n\nTop 5 :\n'
        for i, idx in enumerate(top5):
            texte += f'{i+1}. {noms[idx]:<25} {pred[idx]*100:.1f}%\n'

    return texte, {noms[idx]: float(pred[idx]) for idx in top5}

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🥇 Classificateur CNN — Soulier d'Or
    ### Détection faciale automatique + Test Time Augmentation
    Uploadez une photo d'un footballeur parmi les **22 légendes** du Soulier d'Or !
    """)
    with gr.Row():
        img_in = gr.Image(label='📸 Photo du joueur', type='numpy', height=350)
        with gr.Column():
            txt_out   = gr.Textbox(label='🎯 Résultat de la prédiction', lines=14)
            chart_out = gr.Label(label='📊 Probabilités Top-5', num_top_classes=5)
    gr.Button('🔍 Identifier le joueur', variant='primary', size='lg').click(
        predire, inputs=img_in, outputs=[txt_out, chart_out]
    )
    gr.Markdown('**Modèle :** MobileNetV2 | **Val Accuracy :** ~80% | **Top-3 Acc :** ~91%')

demo.launch(share=True)

---
## 🔄 Rechargement rapide (nouvelle session)
**Utiliser uniquement ces 2 cellules pour recharger le modèle sans réentraîner.**

In [ ]:
# RECHARGEMENT — Cellule A : installation (puis redémarrer le runtime)
!pip install lz4 mtcnn gradio -q
print('✅ Installation terminée — redémarre le runtime !')

In [ ]:
# CELLULE B COMPLÈTE — copie tout et remplace l'ancienne cellule B
import os, json
import numpy as np
import tensorflow as tf
from PIL import Image
import lz4
from mtcnn import MTCNN
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import gradio as gr

# ── Fixer toutes les seeds ──
tf.random.set_seed(42)
np.random.seed(42)

# ── Monter Drive ──
from google.colab import drive
drive.mount('/drive')

# ── Charger modèle ──
MODEL_PATH       = '/drive/MyDrive/soulier_dor_CNN/improved_best.keras'
CLASS_NAMES_PATH = '/drive/MyDrive/soulier_dor_CNN/class_names.json'

model = tf.keras.models.load_model(MODEL_PATH)
with open(CLASS_NAMES_PATH) as f:
    class_names = json.load(f)

noms = [class_names[str(i)].replace('_', ' ').title()
        for i in range(len(class_names))]
print(f"✅ Modèle chargé — {len(noms)} joueurs")
print(f"GPU : {tf.config.list_physical_devices('GPU')}")

# ── Détection faciale ──
detector_g      = MTCNN()
SEUIL_CONFIANCE = 0.40

def extraire_region(image_np):
    res = detector_g.detect_faces(image_np)
    if res:
        best       = max(res, key=lambda x: x['confidence'])
        x, y, w, h = best['box']
        marge      = int(0.35 * max(w, h))
        x1 = max(0, x - marge)
        y1 = max(0, y - marge)
        x2 = min(image_np.shape[1], x + w + marge)
        y2 = min(image_np.shape[0], y + h + marge)
        return image_np[y1:y2, x1:x2], True
    return image_np, False

# ── TTA avec seed fixée ──
def tta_predict(region_np, n=10):
    np.random.seed(42)  # seed fixée à chaque appel
    aug = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        horizontal_flip=True,
        zoom_range=0.1,
        brightness_range=[0.9, 1.1],
        preprocessing_function=preprocess_input
    )
    img_arr = np.expand_dims(
        np.array(Image.fromarray(region_np).resize((160, 160)),
                 dtype=np.float32), 0)

    preds = [model.predict(preprocess_input(img_arr.copy()), verbose=0)[0]]
    gen   = aug.flow(img_arr, batch_size=1, seed=42)
    for _ in range(n - 1):
        preds.append(model.predict(next(gen), verbose=0)[0])

    return np.mean(preds, axis=0)

# ── Prédiction ──
def predire(image):
    img_rgb        = np.array(Image.fromarray(image).convert('RGB'))
    region, trouve = extraire_region(img_rgb)
    note           = "✅ Visage détecté" if trouve else "⚠️ Image entière"

    pred = tta_predict(region, n=10)
    top5 = np.argsort(pred)[::-1][:5]
    conf = pred[top5[0]]

    if conf < SEUIL_CONFIANCE:
        texte = (f"{note}\n\n❓ Joueur non identifié\n"
                 f"Confiance : {conf*100:.1f}%\n\nCandidats :\n")
        for i, idx in enumerate(top5[:3]):
            texte += f"  {i+1}. {noms[idx]:<25} {pred[idx]*100:.1f}%\n"
    else:
        texte  = f"{note}\n\n🏆 {noms[top5[0]]}\n"
        texte += f"Confiance : {conf*100:.1f}%\n\nTop 5 :\n"
        for i, idx in enumerate(top5):
            texte += f"{i+1}. {noms[idx]:<25} {pred[idx]*100:.1f}%\n"

    return texte, {noms[idx]: float(pred[idx]) for idx in top5}

# ── Gradio ──
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🥇 CNN Soulier d'Or — Détection faciale + TTA")
    with gr.Row():
        img_in = gr.Image(label="📸 Photo", type="numpy", height=350)
        with gr.Column():
            txt_out   = gr.Textbox(label="🎯 Résultat", lines=14)
            chart_out = gr.Label(label="📊 Top-5", num_top_classes=5)
    gr.Button("🔍 Identifier", variant="primary").click(
        predire, inputs=img_in, outputs=[txt_out, chart_out]
    )

demo.launch(share=True)

In [ ]:
# Cellule diagnostic — exécute ceci en premier
import os

drive_path = '/drive/MyDrive/soulier_dor_CNN'

if os.path.exists(drive_path):
    print(f"✅ Dossier trouvé : {drive_path}")
    for f in os.listdir(drive_path):
        taille = os.path.getsize(os.path.join(drive_path, f))
        print(f"  {f}  ({taille/1024/1024:.1f} MB)")
else:
    print("❌ Dossier introuvable !")
    print("\nContenu de MyDrive :")
    for f in os.listdir('/drive/MyDrive'):
        print(f"  {f}")